### Build Drivers Dimension

1. Read silver drivers table
2. Read gold ref_nationality_region table
3. Join the data from drivers with ref_nationality_region using nationality
4. Select the required columns:
    * drivers.driver_id
    * drivers.driver_name
    * drivers.date_of_birth
    * drivers.nationality
    * ref_nationality_region.region
5. Write the transformed data to gold dim_drivers table

In [0]:
%run ../00-common/01.environment-configuration

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table =F"{catalog_name}.{gold_schema}.dim_drivers"

In [0]:
drivers_df = spark.read.table(F"{catalog_name}.{silver_schema}.drivers")
ref_nationality_region_df = spark.read.table(F"{catalog_name}.{gold_schema}.ref_nationality_region")



In [0]:
display(drivers_df)

driver_id,date_of_birth,nationality,timestamp,source_file,driver_name
godia,1921-03-21,Spanish,2026-09-13T19:20:29.480755Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Paco Godia
hobbs,1939-06-09,British,2026-09-13T19:20:29.480755Z,dbfs:/Volumes/formula1/landing/files/drivers.json,David Hobbs
karthikeyan,1977-01-14,Indian,2026-09-13T19:20:29.480755Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Narain Karthikeyan
pollet,1922-07-02,French,2026-09-13T19:20:29.480755Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Jacques Pollet
alliot,1954-07-27,French,2026-09-13T19:20:29.480755Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Philippe Alliot
irwin,1942-06-27,British,2026-09-13T19:20:29.480755Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Chris Irwin
magnussen,1973-07-04,Danish,2026-09-13T19:20:29.480755Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Jan Magnussen
rosset,1968-07-27,Brazilian,2026-09-13T19:20:29.480755Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Ricardo Rosset
scarfiotti,1933-10-18,Italian,2026-09-13T19:20:29.480755Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Ludovico Scarfiotti
angelis,1958-03-26,Italian,2026-09-13T19:20:29.480755Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Elio De Angelis


In [0]:
dim_drivers_df =(
    drivers_df
        .join(ref_nationality_region_df, 
                drivers_df.nationality == ref_nationality_region_df.nationality, 
                'left')
        .select(drivers_df.driver_id, 
                drivers_df.driver_name,
                drivers_df.date_of_birth, 
                drivers_df.nationality, 
                ref_nationality_region_df.region.alias("nationality_region")
        )
)         
    

In [0]:
(
    dim_drivers_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(target_table)
)

In [0]:
display(spark.table(target_table))

driver_id,driver_name,date_of_birth,nationality,nationality_region
godia,Paco Godia,1921-03-21,Spanish,Europe
hobbs,David Hobbs,1939-06-09,British,Europe
karthikeyan,Narain Karthikeyan,1977-01-14,Indian,Asia
pollet,Jacques Pollet,1922-07-02,French,Europe
alliot,Philippe Alliot,1954-07-27,French,Europe
irwin,Chris Irwin,1942-06-27,British,Europe
magnussen,Jan Magnussen,1973-07-04,Danish,Europe
rosset,Ricardo Rosset,1968-07-27,Brazilian,South America
scarfiotti,Ludovico Scarfiotti,1933-10-18,Italian,Europe
angelis,Elio De Angelis,1958-03-26,Italian,Europe
